# JIG26_03 — Enterprise Predictive Maintenance (Day 2 Sprint)

## Goal
Build an enterprise-grade early-warning system for campus equipment that predicts failure at **two warning horizons: 6 hours and 24 hours**.

### Mandatory Day 2 additions
- Performance reported separately for **6h** and **24h** horizons.
- Explicit operational costs: **₹10,000 per missed failure** and **₹500 per false alarm** (project assumptions; replace with campus finance data if available).
- Alert thresholds selected on validation data by **minimum expected cost**, not by test-set performance.
- **Machine 100 is held out completely from model training/validation** and evaluated as an unseen equipment unit on December 2015 data.

### Enterprise feature groups
1. Current equipment condition
2. Temporal behavior and degradation
3. Machine-specific anomaly detection
4. Error intelligence
5. Maintenance intelligence
6. Asset context
7. Data quality

> **Important:** all features use information available at or before the prediction timestamp. Future failure information is used only to create the target, never as a predictor.

In [ ]:
import pandas as pd, numpy as np, joblib, json, gc
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

DATA_DIR = Path('/mnt/data') if Path('/mnt/data').exists() else Path('.')
OUT = DATA_DIR / 'predictive_maintenance_day2_artifacts'
OUT.mkdir(exist_ok=True)
SEED = 42
HORIZONS = [6, 24]
HELD_OUT_MACHINE = 100
MISSED_FAILURE_COST = 10_000.0
FALSE_ALARM_COST = 500.0
USE_TRAIN_SAMPLE = True       # Set False on a high-memory machine to use all training rows.
MAX_NEGATIVE_TRAIN_ROWS = 300_000
print('Costs:', MISSED_FAILURE_COST, 'missed failure /', FALSE_ALARM_COST, 'false alarm')

In [ ]:
telemetry = pd.read_csv(DATA_DIR/'PdM_telemetry.csv', parse_dates=['datetime'])
errors = pd.read_csv(DATA_DIR/'PdM_errors.csv', parse_dates=['datetime'])
failures = pd.read_csv(DATA_DIR/'PdM_failures.csv', parse_dates=['datetime'])
machines = pd.read_csv(DATA_DIR/'PdM_machines.csv')
maint = pd.read_csv(DATA_DIR/'PdM_maint.csv', parse_dates=['datetime'])
for d in [telemetry, errors, failures, maint]:
    d.sort_values(['machineID','datetime'], inplace=True)
print('Shapes:', telemetry.shape, errors.shape, failures.shape, machines.shape, maint.shape)
print('Missing values:', {name:int(d.isna().sum().sum()) for name,d in [('telemetry',telemetry),('errors',errors),('failures',failures),('machines',machines),('maintenance',maint)]})
print('Duplicate rows:', {name:int(d.duplicated().sum()) for name,d in [('telemetry',telemetry),('errors',errors),('failures',failures),('machines',machines),('maintenance',maint)]})

## 1. Create two strictly future-looking targets

For every machine timestamp `T`:
- `failure_next_6h = 1` if the next failure occurs **after T and within 6 hours**.
- `failure_next_24h = 1` if the next failure occurs **after T and within 24 hours**.

A failure exactly at `T` is **not** a future failure.

In [ ]:
base = telemetry.sort_values(['datetime','machineID']).reset_index(drop=True)
for h in HORIZONS:
    f = failures[['machineID','datetime']].rename(columns={'datetime':'failure_time'}).sort_values(['failure_time','machineID']).reset_index(drop=True)
    m = pd.merge_asof(base[['datetime','machineID']], f,
                      left_on='datetime', right_on='failure_time', by='machineID',
                      direction='forward', tolerance=pd.Timedelta(hours=h),
                      allow_exact_matches=False)
    base[f'failure_next_{h}h'] = m['failure_time'].notna().astype('int8')
print({h: round(100*base[f'failure_next_{h}h'].mean(),3) for h in HORIZONS})

## 2. Enterprise feature engineering

### Why each group exists
- **Current condition:** what the equipment is doing now.
- **Temporal/degradation:** whether condition is changing or becoming unstable.
- **Machine-specific anomalies:** whether the current state is unusual for this particular asset.
- **Error intelligence:** how recent/frequent/persistent error activity is.
- **Maintenance intelligence:** maintenance recency and frequency.
- **Asset context:** model and age provide lifecycle context.
- **Data quality:** tells the application whether recent sensor data is complete/usable.

The historical baseline is shifted by one row so the current observation cannot define its own anomaly baseline.

In [ ]:
df = base.sort_values(['machineID','datetime']).reset_index(drop=True)
sensors = ['volt','rotate','pressure','vibration']
for c in sensors: df[c] = df[c].astype('float32')
g = df.groupby('machineID', sort=False)

# A. Current condition + temporal degradation
for c in sensors:
    for w,label in [(6,'6h'),(24,'24h')]:
        r = g[c].rolling(w, min_periods=w)
        df[f'{c}_mean_{label}'] = r.mean().reset_index(level=0,drop=True).astype('float32')
        df[f'{c}_std_{label}'] = r.std().reset_index(level=0,drop=True).astype('float32')
        df[f'{c}_min_{label}'] = r.min().reset_index(level=0,drop=True).astype('float32')
        df[f'{c}_max_{label}'] = r.max().reset_index(level=0,drop=True).astype('float32')
    df[f'{c}_change_6h'] = (df[c]-g[c].shift(6)).astype('float32')
    df[f'{c}_change_24h'] = (df[c]-g[c].shift(24)).astype('float32')
    df[f'{c}_trend_24h'] = (df[f'{c}_change_24h']/24.0).astype('float32')

# B. Machine-specific 7-day baseline; shifted to avoid look-ahead
for c in sensors:
    hist = g[c].rolling(168,min_periods=24)
    hm = hist.mean().reset_index(level=0,drop=True).groupby(df.machineID).shift(1)
    hs = hist.std().reset_index(level=0,drop=True).groupby(df.machineID).shift(1)
    df[f'{c}_baseline_7d'] = hm.astype('float32')
    df[f'{c}_zscore_7d'] = ((df[c]-hm)/(hs+1e-6)).astype('float32')

zcols=[f'{c}_zscore_7d' for c in sensors]
df['sensor_anomaly_count']=sum(df[z].abs().gt(3).astype('int8') for z in zcols).astype('int8')
df['sensor_anomaly_flag']=(df.sensor_anomaly_count>0).astype('int8')
df['pressure_vibration_ratio']=(df.pressure/(df.vibration.abs()+1e-3)).astype('float32')
df['rotation_vibration_interaction']=(df.rotate*df.vibration).astype('float32')
df['voltage_rotation_interaction']=(df.volt*df.rotate).astype('float32')

# C. Asset context
df=df.merge(machines[['machineID','model','age']],on='machineID',how='left')
df['age_bucket']=pd.cut(df.age,[-1,5,10,15,100],labels=['0-5','6-10','11-15','16+']).astype(str)

# D. Error intelligence
eh=errors.assign(event=1).groupby(['machineID','datetime'],as_index=False).event.sum().rename(columns={'event':'error_count'})
df=df.merge(eh,on=['machineID','datetime'],how='left'); df['error_count']=df.error_count.fillna(0).astype('float32')
et=errors.assign(event=1).pivot_table(index=['machineID','datetime'],columns='errorID',values='event',aggfunc='sum',fill_value=0).reset_index()
etcols=[c for c in et.columns if c not in ['machineID','datetime']]
et['distinct_error_types']=et[etcols].gt(0).sum(axis=1).astype('int8')
df=df.merge(et[['machineID','datetime','distinct_error_types']],on=['machineID','datetime'],how='left'); df['distinct_error_types']=df.distinct_error_types.fillna(0).astype('int8')
df.sort_values(['machineID','datetime'],inplace=True); g=df.groupby('machineID',sort=False)
for w,label in [(6,'6h'),(24,'24h'),(168,'7d')]:
    df[f'errors_{label}']=g.error_count.rolling(w,min_periods=1).sum().reset_index(level=0,drop=True).astype('float32')
df['distinct_error_types_24h']=g.distinct_error_types.rolling(24,min_periods=1).max().reset_index(level=0,drop=True).astype('int8')
df['error_burst_flag']=(df.errors_24h>=2).astype('int8')
last_err=pd.merge_asof(df[['machineID','datetime']].sort_values(['datetime','machineID']),
                        errors[['machineID','datetime']].rename(columns={'datetime':'error_time'}).sort_values(['error_time','machineID']),
                        left_on='datetime',right_on='error_time',by='machineID',direction='backward',allow_exact_matches=True)
df=df.merge(last_err[['machineID','datetime','error_time']],on=['machineID','datetime'],how='left')
df['hours_since_error']=((df.datetime-df.error_time).dt.total_seconds()/3600).astype('float32')
df.drop(columns='error_time',inplace=True)

# E. Maintenance intelligence
mh=maint.assign(event=1).groupby(['machineID','datetime'],as_index=False).event.sum().rename(columns={'event':'maintenance_count'})
df=df.merge(mh,on=['machineID','datetime'],how='left'); df['maintenance_count']=df.maintenance_count.fillna(0).astype('float32')
df.sort_values(['machineID','datetime'],inplace=True); g=df.groupby('machineID',sort=False)
for w,label in [(168,'7d'),(720,'30d'),(2160,'90d')]:
    df[f'maintenance_count_{label}']=g.maintenance_count.rolling(w,min_periods=1).sum().reset_index(level=0,drop=True).astype('float32')
last_maint=pd.merge_asof(df[['machineID','datetime']].sort_values(['datetime','machineID']),
                         maint[['machineID','datetime']].rename(columns={'datetime':'maintenance_time'}).sort_values(['maintenance_time','machineID']),
                         left_on='datetime',right_on='maintenance_time',by='machineID',direction='backward',allow_exact_matches=True)
df=df.merge(last_maint[['machineID','datetime','maintenance_time']],on=['machineID','datetime'],how='left')
df['hours_since_maintenance']=((df.datetime-df.maintenance_time).dt.total_seconds()/3600).astype('float32')
df['days_since_maintenance']=df.hours_since_maintenance/24.0
df['maintenance_overdue_30d']=(df.hours_since_maintenance>720).astype('int8')
df.drop(columns='maintenance_time',inplace=True)
mc=maint.assign(event=1).pivot_table(index=['machineID','datetime'],columns='comp',values='event',aggfunc='sum',fill_value=0).reset_index()
for c in [c for c in mc.columns if c not in ['machineID','datetime']]:
    name=f'maint_comp_{c}_event'; mc.rename(columns={c:name},inplace=True)
df=df.merge(mc,on=['machineID','datetime'],how='left')
df.sort_values(['machineID','datetime'],inplace=True); g=df.groupby('machineID',sort=False)
for c in [c for c in mc.columns if c.startswith('maint_comp_')]:
    df[c]=df[c].fillna(0).astype('float32')
    df[c.replace('_event','_30d')]=g[c].rolling(720,min_periods=1).sum().reset_index(level=0,drop=True).astype('float32')

# F. Data quality: historical coverage and stale/missing indicators.
for c in sensors:
    valid=g[c].rolling(24,min_periods=1).count().reset_index(level=0,drop=True)
    df[f'{c}_coverage_24h']=(valid/24.0).astype('float32')
    df[f'{c}_low_coverage_flag']=(valid<18).astype('int8')
df['sensor_coverage_24h']=df[[f'{c}_coverage_24h' for c in sensors]].mean(axis=1).astype('float32')
df['sensor_low_coverage_flag']=(df.sensor_coverage_24h<0.75).astype('int8')
# Historical dataset has an observation exactly at each telemetry timestamp; external ingestion latency is unavailable.
df['data_freshness_hours']=0.0

# 24h warm-up is required for temporal features.
df=df.dropna(subset=['vibration_mean_24h','vibration_std_24h']).reset_index(drop=True)
exclude={'datetime','machineID','failure_next_6h','failure_next_24h'}
feature_cols=[c for c in df.columns if c not in exclude and c!='event']
print('Rows:',len(df),'Engineered features:',len(feature_cols))

## 3. Time split + unseen-equipment split

We keep the normal chronological split for operational realism:
- Train: before **2015-10-01**
- Validation: **2015-10-01 to 2015-11-30**
- Test: **2015-12-01 onward**

Additionally, **Machine 100 is excluded from train and validation entirely**. Its December observations form a dedicated unseen-equipment test. This directly tests whether the learned patterns generalize to equipment the model never saw during training.

In [ ]:
train_end=pd.Timestamp('2015-10-01'); val_end=pd.Timestamp('2015-12-01')
seen=df[df.machineID!=HELD_OUT_MACHINE]
unseen=df[df.machineID==HELD_OUT_MACHINE]
train=seen[seen.datetime<train_end].copy()
val=seen[(seen.datetime>=train_end)&(seen.datetime<val_end)].copy()
test=seen[seen.datetime>=val_end].copy()
unseen_test=unseen[unseen.datetime>=val_end].copy()
print('Train:',len(train),'Validation:',len(val),'Test:',len(test),'Unseen-machine test:',len(unseen_test))
print('Unseen machine failures:', {h:int(unseen_test[f'failure_next_{h}h'].sum()) for h in HORIZONS})

In [ ]:
# Optional memory-safe training sample: retain every positive 24h case and sample negatives.
if USE_TRAIN_SAMPLE:
    rng=np.random.default_rng(SEED)
    pos_idx=np.flatnonzero(train.failure_next_24h.to_numpy()==1)
    neg_idx=np.flatnonzero(train.failure_next_24h.to_numpy()==0)
    neg_keep=rng.choice(neg_idx,size=min(MAX_NEGATIVE_TRAIN_ROWS,len(neg_idx)),replace=False)
    model_idx=np.sort(np.concatenate([pos_idx,neg_keep]))
    train_model=train.iloc[model_idx].copy()
else:
    train_model=train.copy()
print('Rows used for model fitting:',len(train_model))

In [ ]:
num=[c for c in feature_cols if c not in ['model','age_bucket']]
cat=['model','age_bucket']
pre=ColumnTransformer([
    ('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),
    ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat)
])
Xtr=pre.fit_transform(train_model[feature_cols]).astype('float32')
Xv=pre.transform(val[feature_cols]).astype('float32')
Xte=pre.transform(test[feature_cols]).astype('float32')
Xun=pre.transform(unseen_test[feature_cols]).astype('float32')
joblib.dump(pre,OUT/'enterprise_preprocessor.joblib')
print('Model matrix:',Xtr.shape)

## 4. Models

For every warning horizon we train:
- **Logistic Regression** — transparent baseline.
- **ANN** — nonlinear model with hidden layers `(64, 32, 16)`.

Class imbalance is handled using class-weighted Logistic Regression and sample weights for the ANN.

In [ ]:
def metrics_at_threshold(y,p,t,miss=MISSED_FAILURE_COST,false=FALSE_ALARM_COST):
    pred=(p>=t).astype('int8')
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'threshold':float(t),'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),
            'f1':f1_score(y,pred,zero_division=0),'false_alarm_rate':fp/(fp+tn) if fp+tn else 0,
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
            'expected_cost':fn*miss+fp*false}

def tune_cost(y,p):
    grid=np.arange(0.05,0.951,0.01)
    table=pd.DataFrame([metrics_at_threshold(y,p,t) for t in grid])
    best=table.sort_values(['expected_cost','f1','recall','precision'],ascending=[True,False,False,False]).iloc[0]
    return table,best

all_metrics=[]; thresholds=[]; prediction_frames={}
for h in HORIZONS:
    ytr=train_model[f'failure_next_{h}h'].to_numpy()
    yv=val[f'failure_next_{h}h'].to_numpy(); yt=test[f'failure_next_{h}h'].to_numpy(); yu=unseen_test[f'failure_next_{h}h'].to_numpy()
    print(f'\n===== {h}-HOUR HORIZON =====')

    # Baseline
    log=LogisticRegression(max_iter=500,class_weight='balanced',random_state=SEED)
    log.fit(Xtr,ytr)
    pv=log.predict_proba(Xv)[:,1]; pt=log.predict_proba(Xte)[:,1]; pu=log.predict_proba(Xun)[:,1]
    table,best=tune_cost(yv,pv)
    thresholds.append({'horizon':h,'model':'Logistic Regression','threshold':float(best.threshold),'validation_expected_cost':float(best.expected_cost),
                       'missed_failure_cost':MISSED_FAILURE_COST,'false_alarm_cost':FALSE_ALARM_COST})
    for split,yy,pp in [('test',yt,pt),('unseen_machine',yu,pu)]:
        m=metrics_at_threshold(yy,pp,best.threshold); m.update(model='Logistic Regression',horizon=h,split=split,
             roc_auc=roc_auc_score(yy,pp),pr_auc=average_precision_score(yy,pp)); all_metrics.append(m)
    joblib.dump(log,OUT/f'logistic_{h}h.joblib'); table.to_csv(OUT/f'logistic_thresholds_{h}h.csv',index=False)

    # ANN
    ann=MLPClassifier(hidden_layer_sizes=(64,32,16),activation='relu',solver='adam',alpha=1e-4,batch_size=4096,
                      learning_rate_init=1e-3,max_iter=15,early_stopping=True,validation_fraction=0.1,
                      n_iter_no_change=3,random_state=SEED)
    sw=compute_sample_weight(class_weight='balanced',y=ytr)
    ann.fit(Xtr,ytr,sample_weight=sw)
    pv=ann.predict_proba(Xv)[:,1]; pt=ann.predict_proba(Xte)[:,1]; pu=ann.predict_proba(Xun)[:,1]
    table,best=tune_cost(yv,pv)
    thresholds.append({'horizon':h,'model':'ANN','threshold':float(best.threshold),'validation_expected_cost':float(best.expected_cost),
                       'missed_failure_cost':MISSED_FAILURE_COST,'false_alarm_cost':FALSE_ALARM_COST})
    for split,yy,pp in [('test',yt,pt),('unseen_machine',yu,pu)]:
        m=metrics_at_threshold(yy,pp,best.threshold); m.update(model='ANN',horizon=h,split=split,
             roc_auc=roc_auc_score(yy,pp),pr_auc=average_precision_score(yu if split=='unseen_machine' else yt,pp)); all_metrics.append(m)
    joblib.dump(ann,OUT/f'ann_{h}h.joblib'); table.to_csv(OUT/f'ann_thresholds_{h}h.csv',index=False)
    prediction_frames[h]=pd.DataFrame({'datetime':test.datetime,'machineID':test.machineID,f'actual_failure_{h}h':yt,
                                       'logistic_probability':log.predict_proba(Xte)[:,1],'ann_probability':pt})
    prediction_frames[h].to_csv(OUT/f'test_predictions_{h}h.csv',index=False)
    pd.DataFrame({'datetime':unseen_test.datetime,'machineID':unseen_test.machineID,f'actual_failure_{h}h':yu,
                  'logistic_probability':log.predict_proba(Xun)[:,1],'ann_probability':pu}).to_csv(OUT/f'unseen_machine_predictions_{h}h.csv',index=False)

results=pd.DataFrame(all_metrics); thresholds_df=pd.DataFrame(thresholds)
results.to_csv(OUT/'metrics_all.csv',index=False); thresholds_df.to_csv(OUT/'cost_thresholds.csv',index=False)
display(thresholds_df); display(results)

## 5. How the alert threshold is justified by cost

We explicitly assume:

- **Missed failure (FN): ₹10,000** — represents the potential combined cost of downtime, emergency repair and disruption.
- **False alarm (FP): ₹500** — represents an illustrative cost for unnecessary inspection/maintenance attention.

For each candidate probability threshold `t`, we calculate:

**Expected operational cost = (FN × ₹10,000) + (FP × ₹500)**

The threshold is selected using the **validation set only**, minimizing this cost. The test set is never used to select the threshold.

These values are project assumptions for the sprint; in a real campus deployment they should be replaced with finance/maintenance estimates.

## 6. Unseen equipment evaluation

Machine 100 is never used in model fitting or validation. Its December observations are evaluated after the thresholds are frozen.

Report for each horizon:
- Precision
- Recall
- F1
- False-alarm rate
- TP / FP / FN / TN
- Number of actual failures in the held-out machine

**Interpretation rule:** because the held-out machine has very few actual failures, its metrics are a generalization check rather than a statistically stable estimate of fleet-wide performance. Report the counts alongside the percentages.

In [ ]:
print('UNSEEN MACHINE GENERALIZATION (Machine 100)')
print(results[results.split=='unseen_machine'][['horizon','model','precision','recall','f1','false_alarm_rate','TP','FP','FN','TN']].to_string(index=False))
print('\nFLEET TEST PERFORMANCE')
print(results[results.split=='test'][['horizon','model','precision','recall','f1','false_alarm_rate','TP','FP','FN','TN']].to_string(index=False))

## 7. Maintenance Priority and Explainability

The ML model estimates failure risk. A separate operational layer converts that risk into **LOW / MEDIUM / HIGH** maintenance priority.

For each horizon, HIGH priority is driven by the cost-optimized ANN threshold for the operationally selected horizon (24h in the main dashboard), while MEDIUM can also be triggered by supporting warning signals such as recent errors or strong sensor anomalies.

The dashboard should show:
- failure risk at 6h and 24h
- selected alert threshold
- equipment status
- maintenance priority
- sensor trends
- recent errors
- maintenance recency
- data-quality status
- human-readable reasons

## 8. Judge-ready explanation

> **“Our system now forecasts failure at two warning horizons, 6 hours and 24 hours, so the maintenance team can see both near-term and longer lead-time risk. We report performance separately for each horizon. We also made the threshold cost-sensitive: a missed failure is assigned ₹10,000 and a false alarm ₹500, so we select the threshold on validation data by minimizing expected operational cost. Finally, we hold out Machine 100 completely from training and validation and evaluate it separately as unseen equipment. This tests whether our enterprise features generalize beyond machines the model has already seen.”**